In [96]:
import os
from datetime import  date
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### *Exercise 1 — Load PDFs*

Use PyPDFLoader from LangChain to load 2 different PDF files of your choice (e.g., a textbook chapter, a research paper, personal notes).

In [20]:
loader = PyPDFLoader(file_path="UNIT I.pdf")
page_1= loader.load()
loader_2= PyPDFLoader(file_path="UNIT II.pdf")
page_2= loader_2.load()

print(f"page_1 : {len(page_1)} pages")
print(f"page_2 : {len(page_2)} pages")

print(f"page_1 length: {len(page_1[0].page_content)}")
print(f"page_2 length: {len(page_2[0].page_content)}")


print(page_1[0].page_content)
print(page_2[0].page_content)



page_1 : 24 pages
page_2 : 35 pages
page_1 length: 1267
page_2 length: 1673
Natural Language Processing (NLP) 
24BAM6C10 
 
UNIT I 
 
 
1. Introduction: Computing with Language 
Natural Language Processing (NLP) is  an AI field combining computer science, 
linguistics, and machine learning to help computers understand, interpret, and generate human 
language (text and speech). It bridges the gap between human communication and machine 
 
 It focuses on enabling computers to process, understand, and generate human language. 
The phrase “Computing with Language”  refers to the computational treatment of linguistic 
data—how computers can be programmed to handle text and speech in natural languages. 
In the modern era, language data is abundant in the form of emails, web pages, news articles, 
blogs, and social media posts. The challenge is to convert this raw, unstructured text into 
structured data that computers can understand and use for various applications such as  
• Search engines

In [29]:
pages = page_1 + page_2
print(f"total : {len(pages)} pages")
print(pages[25].metadata)

total : 59 pages
{'producer': 'Microsoft® Word 2024', 'creator': 'Microsoft® Word 2024', 'creationdate': '2026-02-11T21:27:04+05:30', 'author': 'DELL', 'moddate': '2026-02-11T21:27:04+05:30', 'source': 'UNIT II.pdf', 'total_pages': 35, 'page': 1, 'page_label': '2'}


### *Exercise 2 — Split into Chunks*

Use RecursiveCharacterTextSplitter with chunk_size=8000 and chunk_overlap=150 to split the loaded documents into chunks.

In [36]:

text_split = RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=150)
chunk = text_split.split_documents(pages)
print(f"total : {len(pages)} pages")
print(f"length of chunk after splitting : {len(chunk)}")


total : 59 pages
length of chunk after splitting : 128


In [40]:
print(f"Chunk 1 : \n\n{chunk[0].page_content}\n")
print(f"Chunk 127 : \n\n{chunk[127].page_content}")

Chunk 1 : 

Natural Language Processing (NLP) 
24BAM6C10 
 
UNIT I 
 
 
1. Introduction: Computing with Language 
Natural Language Processing (NLP) is  an AI field combining computer science, 
linguistics, and machine learning to help computers understand, interpret, and generate human 
language (text and speech). It bridges the gap between human communication and machine 
 
 It focuses on enabling computers to process, understand, and generate human language. 
The phrase “Computing with Language”  refers to the computational treatment of linguistic 
data—how computers can be programmed to handle text and speech in natural languages. 
In the modern era, language data is abundant in the form of emails, web pages, news articles,

Chunk 127 : 

4. Describe text normalization techniques including lowercasing, stemming, and 
lemmatization with examples. 
5. Explain segmentation as a search problem and discuss objective functions. 
6. Compare procedural and declarative programming styles wit

In [43]:
print("chunk_1 : \n\n",chunk[0].page_content[:100],"\n")
print("chunk_2 : \n\n",chunk[1].page_content[-50:],"\n")
print("chunk_100 : \n\n",chunk[100].page_content[:100],"\n")


chunk_1 : 

 Natural Language Processing (NLP) 
24BAM6C10 
 
UNIT I 
 
 
1. Introduction: Computing with Language 

chunk_2 : 

 ing from text 
3. Generating natural language text 

chunk_100 : 

 o Iterated 
• Length can be found using len() 
Types of Sequence Objects 
Sequence Type Description  



### *Exercise 3 — Attach Metadata*

For each chunk, attach the following metadata fields:
- filename
- page_number
- upload_date
- source_type (e.g., "textbook", "paper", "notes" — your choice)


In [56]:
print(f"old metadata for page: {pages[0].metadata}")


pdf_metadata ={
    "UNIT I.pdf": {"source":"document"},
    "UNIT II.pdf":{"source":"paper"}
}

today = date.today()
print(f"today's date: {today}")

for chunks in chunk :
    source_path =chunks.metadata.get("source", "")
    file_path = os.path.basename(source_path)

    chunks.metadata["File_name"] = file_path
    chunks.metadata["page_number"]= chunks.metadata.get("page", 0)
    chunks.metadata["upload_date"] = today
    chunks.metadata["source_type"] = pdf_metadata.get(file_path, {}).get("source", "unknown")



old metadata for page: {'producer': 'Microsoft® Word 2024', 'creator': 'Microsoft® Word 2024', 'creationdate': '2026-02-05T15:40:35+05:30', 'author': 'Un-named', 'moddate': '2026-02-05T15:40:35+05:30', 'source': 'UNIT I.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}
today's date: 2026-04-15


In [60]:
file_path = set([c.metadata["File_name"] for c in chunk])
page_number = set([c.metadata["page_number"] for c in chunk])
upload_date = set([c.metadata["upload_date"] for c in chunk])
source_type = set([c.metadata["source_type"] for c in chunk])

print(f"file_path: {file_path}\n")
print(f"page_number: {page_number}\n")      
print(f"upload_date: {upload_date}\n")
print(f"source_type: {source_type}\n")


file_path: {'UNIT II.pdf', 'UNIT I.pdf'}

page_number: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34}

upload_date: {datetime.date(2026, 4, 15)}

source_type: {'document', 'paper'}



### Core idea used in RAG systems
### *Exercise 4 — Build a Filter Function*

Write a function filter_chunks(chunks, **filters) that returns only the chunks matching the given metadata key-value pairs.

Example usage: filter_chunks(chunks, filename="paper1.pdf", page_number=3)


In [67]:
def filter(chunks , **filter):
    result = []
    for chunk in chunks :
        match = True
        for key , value in filter.items():
            if chunk.metadata.get(key) != value:
                match = False
                break
        if match:
            result.append(chunk)
    return result
    

### *Exercise 5 — Test Your Filters*

Use the functions created and test your data

## UNIT I.pdf

In [95]:
filtered_chunks = filter(chunk, File_name="UNIT I.pdf", source_type="document")
print(f"filtered_chunks: {len(filtered_chunks)}")  
for i,j in enumerate(filtered_chunks):
        print(f"---chunks ({i}) : ({len(j.page_content)}) characters\n") 
        print(j.page_content[:200],"..\n")

        
print(filtered_chunks[56].metadata)

filtered_chunks: 57
---chunks (0) : (724) characters

Natural Language Processing (NLP) 
24BAM6C10 
 
UNIT I 
 
 
1. Introduction: Computing with Language 
Natural Language Processing (NLP) is  an AI field combining computer science, 
linguistics, and ma ..

---chunks (1) : (636) characters

In the modern era, language data is abundant in the form of emails, web pages, news articles, 
blogs, and social media posts. The challenge is to convert this raw, unstructured text into 
structured d ..

---chunks (2) : (753) characters

4. Building systems that can understand or produce language automatically 
Python has emerged as a preferred programming language for NLP due to its simplicity and 
the availability of powerful librar ..

---chunks (3) : (779) characters

• Tokenization: Breaking text into words or sentences. 
• Stopword Removal: Removing common words (like "the", "is", "at") that add little 
meaning. 
• Stemming & Lemmatization:  Reducing words to the ..

---chunks (4) : (209) ch

## UNIT-2.pdf

In [94]:
filter_chunks_2 = filter(chunk, File_name="UNIT II.pdf", source_type="paper", page_number=5)
print(f"filtered_chunks: {len(filter_chunks_2)}\n")
# print(filter_chunks_2[0].page_content)


for i , e in enumerate(filter_chunks_2):
    print(f"---chunks ({i}) : ({len(e.page_content)}) characters\n")
    print(e.page_content[:200],"..\n")

filtered_chunks: 2

---chunks (0) : (777) characters

o Latin characters: ø 
o Hungarian: ő 
o Spanish and Breton: ñ 
o Czech and Slovak: ň 
• Indian languages: 
o Indic scripts (Devanagari, Tamil, Malayalam, etc.) 
As a result, ASCII-based systems fail  ..

---chunks (1) : (386) characters

U+XXXX   (hexadecimal) 
Examples 
• U+0D05 → അ (Malayalam letter) 
• U+0B85 → அ (Tamil letter) 
This numeric value uniquely identifies the character, independent of platform or language. 
Characters v ..

